# DeepSets Inference Speed Analysis

This notebook analyzes the inference speed of trained DeepSets models across different code distances (d=3-13).

**Goals:**
- Measure inference time breakdown (coordinate conversion vs forward pass)
- Identify bottlenecks in the inference pipeline
- Test scaling behavior with code distance
- Use minimal data for fast debugging

## Setup

In [ ]:
import sys
import time
from pathlib import Path

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/deepsets/speedup -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Import from benchmark_models.py
from benchmark_models import DeepSets, DeepSetsModel, SurfaceCodeSampler

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Paths
MODELS_DIR = BASE_PATH / "deepsets" / "extrapolation" / "models" / "revised_training"
print(f"\nModels directory: {MODELS_DIR}")
print(f"Models exist: {MODELS_DIR.exists()}")

## Configuration

In [ ]:
# =============================================================================
# MINIMAL CONFIG FOR FAST DEBUGGING
# =============================================================================

# Distances to test (odd numbers for surface code)
TEST_DISTANCES = [3, 5, 7, 9, 11, 13]

# Small sample sizes for fast iteration
NUM_SAMPLES = 100  # Small for debugging
BATCH_SIZE = 32

# Error rate for data generation
P_VALUE = 0.005

# Number of timing trials for averaging
NUM_TRIALS = 3

print(f"Test Configuration:")
print(f"  Distances: {TEST_DISTANCES}")
print(f"  Samples per distance: {NUM_SAMPLES}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  P value: {P_VALUE}")
print(f"  Timing trials: {NUM_TRIALS}")

## Load a Trained Model

In [ ]:
# List available models
available_models = [f.stem for f in MODELS_DIR.glob("*.pt") if "checkpoint" not in f.stem]
print(f"Available models: {len(available_models)}")
for m in available_models[:5]:
    print(f"  - {m}")
if len(available_models) > 5:
    print(f"  ... and {len(available_models) - 5} more")

In [ ]:
# Load one model for testing
MODEL_NAME = "equal_333333"  # Use the balanced baseline model
model_path = MODELS_DIR / f"{MODEL_NAME}.pt"

print(f"Loading model: {model_path}")

# Load the saved model
save_dict = torch.load(model_path, map_location=device, weights_only=False)
config = save_dict['config']

print(f"\nModel config:")
for k, v in config.items():
    print(f"  {k}: {v}")

# Create DeepSets wrapper and load weights
deepsets = DeepSets(
    nickname=MODEL_NAME,
    phi_hidden=config['phi_hidden'],
    rho_hidden=config['rho_hidden'],
    pool=config['pool'],
    dropout=config['dropout'],
    device=device,
    base_path=BASE_PATH
)
deepsets.model.load_state_dict(save_dict['state_dict'])
deepsets.model.eval()

print(f"\nModel loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in deepsets.model.parameters()):,}")

## Data Generation Speed

In [ ]:
# Test data generation speed for each distance
sampler = SurfaceCodeSampler(p=P_VALUE, device=device)

data_gen_times = {}
num_detectors = {}

print("Data generation speed test:")
print(f"{'Distance':<10} {'Detectors':<12} {'Time (ms)':<12} {'Samples/sec':<12}")
print("-" * 50)

for d in TEST_DISTANCES:
    times = []
    for _ in range(NUM_TRIALS):
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        start = time.perf_counter()
        
        detections, labels = sampler.sample(d=d, num_samples=NUM_SAMPLES)
        
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        elapsed = time.perf_counter() - start
        times.append(elapsed)
    
    avg_time = np.mean(times)
    data_gen_times[d] = avg_time
    num_detectors[d] = detections.shape[1]
    
    samples_per_sec = NUM_SAMPLES / avg_time
    print(f"{d:<10} {num_detectors[d]:<12} {avg_time*1000:<12.2f} {samples_per_sec:<12.0f}")

## Inference Pipeline Breakdown

Measure each step of the inference pipeline:
1. Coordinate conversion (`_syndromes_to_coords`)
2. Model forward pass

In [ ]:
def time_inference_breakdown(deepsets, detections, d, num_trials=3):
    """
    Time each step of the inference pipeline.
    
    Returns:
        dict with timing for each step
    """
    coord_times = []
    forward_times = []
    total_times = []
    
    detections = detections.float().to(deepsets.device)
    
    for _ in range(num_trials):
        # Warm up coordinate cache
        _ = deepsets._get_detector_coordinates(d)
        
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        
        # Time coordinate conversion
        t0 = time.perf_counter()
        coords, counts = deepsets._syndromes_to_coords(detections, d)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t1 = time.perf_counter()
        
        # Time forward pass
        with torch.no_grad():
            _ = deepsets.model(coords, counts)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t2 = time.perf_counter()
        
        coord_times.append(t1 - t0)
        forward_times.append(t2 - t1)
        total_times.append(t2 - t0)
    
    return {
        'coord_conversion_ms': np.mean(coord_times) * 1000,
        'forward_pass_ms': np.mean(forward_times) * 1000,
        'total_ms': np.mean(total_times) * 1000,
        'coord_pct': np.mean(coord_times) / np.mean(total_times) * 100,
        'forward_pct': np.mean(forward_times) / np.mean(total_times) * 100,
    }

In [ ]:
# Run inference breakdown for each distance
results = []

print("Inference Pipeline Breakdown:")
print(f"{'Distance':<8} {'Detectors':<10} {'Coord (ms)':<12} {'Forward (ms)':<12} {'Total (ms)':<12} {'Coord %':<10}")
print("-" * 70)

for d in TEST_DISTANCES:
    # Generate test data
    detections, labels = sampler.sample(d=d, num_samples=NUM_SAMPLES)
    
    # Time inference breakdown
    timing = time_inference_breakdown(deepsets, detections, d, num_trials=NUM_TRIALS)
    
    results.append({
        'distance': d,
        'num_detectors': detections.shape[1],
        **timing
    })
    
    print(f"{d:<8} {detections.shape[1]:<10} {timing['coord_conversion_ms']:<12.3f} "
          f"{timing['forward_pass_ms']:<12.3f} {timing['total_ms']:<12.3f} {timing['coord_pct']:<10.1f}")

results_df = pd.DataFrame(results)
results_df

## Batched vs Full Inference Comparison

In [ ]:
def time_batched_inference(deepsets, detections, d, batch_size=32, num_trials=3):
    """
    Time batched inference (processing in chunks).
    """
    times = []
    detections = detections.float().to(deepsets.device)
    n_samples = detections.shape[0]
    
    for _ in range(num_trials):
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        start = time.perf_counter()
        
        all_preds = []
        for i in range(0, n_samples, batch_size):
            batch = detections[i:i+batch_size]
            preds = deepsets.predict(batch, d)
            all_preds.append(preds)
        
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        elapsed = time.perf_counter() - start
        times.append(elapsed)
    
    return np.mean(times) * 1000  # Return in ms


def time_full_batch_inference(deepsets, detections, d, num_trials=3):
    """
    Time inference with all samples in a single batch.
    """
    times = []
    detections = detections.float().to(deepsets.device)
    
    for _ in range(num_trials):
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        start = time.perf_counter()
        
        preds = deepsets.predict(detections, d)
        
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        elapsed = time.perf_counter() - start
        times.append(elapsed)
    
    return np.mean(times) * 1000  # Return in ms

In [ ]:
# Compare batched vs full batch inference
batch_comparison = []

print("Batched vs Full Batch Inference:")
print(f"{'Distance':<8} {'Batched (ms)':<15} {'Full (ms)':<15} {'Speedup':<10}")
print("-" * 50)

for d in TEST_DISTANCES:
    detections, labels = sampler.sample(d=d, num_samples=NUM_SAMPLES)
    
    batched_time = time_batched_inference(deepsets, detections, d, batch_size=BATCH_SIZE, num_trials=NUM_TRIALS)
    full_time = time_full_batch_inference(deepsets, detections, d, num_trials=NUM_TRIALS)
    
    speedup = batched_time / full_time
    
    batch_comparison.append({
        'distance': d,
        'batched_ms': batched_time,
        'full_batch_ms': full_time,
        'speedup': speedup
    })
    
    print(f"{d:<8} {batched_time:<15.3f} {full_time:<15.3f} {speedup:<10.2f}x")

batch_df = pd.DataFrame(batch_comparison)
batch_df

## Coordinate Conversion Analysis

The `_syndromes_to_coords` function may be a bottleneck. Let's analyze it in detail.

In [ ]:
# Analyze coordinate conversion step by step
def detailed_coord_timing(deepsets, detections, d, num_trials=5):
    """
    Break down coordinate conversion into sub-steps.
    """
    detections = detections.float().to(deepsets.device)
    batch_size = detections.shape[0]
    
    # Warm up cache
    detector_coords = deepsets._get_detector_coordinates(d)
    
    get_coords_times = []
    mask_times = []
    nonzero_times = []
    scatter_times = []
    
    for _ in range(num_trials):
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        
        # Step 1: Get detector coordinates (cached)
        t0 = time.perf_counter()
        detector_coords = deepsets._get_detector_coordinates(d)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t1 = time.perf_counter()
        
        # Step 2: Create fired mask and count
        fired_mask = detections > 0.5
        fired_counts = fired_mask.sum(dim=1).long()
        max_fired = max(int(fired_counts.max().item()), 1)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t2 = time.perf_counter()
        
        # Step 3: Get nonzero indices
        batch_indices, detector_indices = fired_mask.nonzero(as_tuple=True)
        fired_coords = detector_coords[detector_indices]
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t3 = time.perf_counter()
        
        # Step 4: Compute positions and scatter
        cumsum = torch.zeros(batch_size + 1, dtype=torch.long, device=deepsets.device)
        cumsum[1:] = fired_counts.cumsum(dim=0)
        positions = torch.arange(len(batch_indices), device=deepsets.device) - cumsum[batch_indices]
        coords = torch.zeros(batch_size, max_fired, 3, device=deepsets.device)
        coords[batch_indices, positions] = fired_coords
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t4 = time.perf_counter()
        
        get_coords_times.append(t1 - t0)
        mask_times.append(t2 - t1)
        nonzero_times.append(t3 - t2)
        scatter_times.append(t4 - t3)
    
    return {
        'get_coords_ms': np.mean(get_coords_times) * 1000,
        'mask_count_ms': np.mean(mask_times) * 1000,
        'nonzero_ms': np.mean(nonzero_times) * 1000,
        'scatter_ms': np.mean(scatter_times) * 1000,
        'total_ms': (np.mean(get_coords_times) + np.mean(mask_times) + 
                     np.mean(nonzero_times) + np.mean(scatter_times)) * 1000,
        'max_fired': max_fired,
        'avg_fired': fired_counts.float().mean().item()
    }

In [ ]:
# Run detailed timing for each distance
coord_analysis = []

print("Coordinate Conversion Breakdown:")
print(f"{'d':<5} {'GetCoords':<12} {'Mask':<12} {'Nonzero':<12} {'Scatter':<12} {'AvgFired':<10}")
print("-" * 70)

for d in TEST_DISTANCES:
    detections, labels = sampler.sample(d=d, num_samples=NUM_SAMPLES)
    timing = detailed_coord_timing(deepsets, detections, d, num_trials=NUM_TRIALS)
    
    coord_analysis.append({
        'distance': d,
        **timing
    })
    
    print(f"{d:<5} {timing['get_coords_ms']:<12.3f} {timing['mask_count_ms']:<12.3f} "
          f"{timing['nonzero_ms']:<12.3f} {timing['scatter_ms']:<12.3f} {timing['avg_fired']:<10.1f}")

coord_df = pd.DataFrame(coord_analysis)
coord_df

## Throughput Analysis

In [ ]:
# Calculate throughput (samples per second)
throughput_results = []

print("Throughput Analysis:")
print(f"{'Distance':<10} {'Samples/sec':<15} {'Detectors':<12} {'ms/sample':<12}")
print("-" * 50)

for d in TEST_DISTANCES:
    detections, labels = sampler.sample(d=d, num_samples=NUM_SAMPLES)
    
    # Time full batch inference
    times = []
    for _ in range(NUM_TRIALS):
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        start = time.perf_counter()
        _ = deepsets.predict(detections, d)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        times.append(time.perf_counter() - start)
    
    avg_time = np.mean(times)
    samples_per_sec = NUM_SAMPLES / avg_time
    ms_per_sample = avg_time * 1000 / NUM_SAMPLES
    
    throughput_results.append({
        'distance': d,
        'num_detectors': detections.shape[1],
        'samples_per_sec': samples_per_sec,
        'ms_per_sample': ms_per_sample
    })
    
    print(f"{d:<10} {samples_per_sec:<15.0f} {detections.shape[1]:<12} {ms_per_sample:<12.4f}")

throughput_df = pd.DataFrame(throughput_results)
throughput_df

## Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Inference time vs distance
ax1 = axes[0, 0]
ax1.plot(results_df['distance'], results_df['total_ms'], 'o-', label='Total', linewidth=2, markersize=8)
ax1.plot(results_df['distance'], results_df['coord_conversion_ms'], 's--', label='Coord Conv', linewidth=2, markersize=8)
ax1.plot(results_df['distance'], results_df['forward_pass_ms'], '^:', label='Forward Pass', linewidth=2, markersize=8)
ax1.set_xlabel('Code Distance')
ax1.set_ylabel('Time (ms)')
ax1.set_title(f'Inference Time Breakdown (n={NUM_SAMPLES})')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Time proportion
ax2 = axes[0, 1]
ax2.bar(results_df['distance'] - 0.2, results_df['coord_pct'], width=0.4, label='Coord Conv')
ax2.bar(results_df['distance'] + 0.2, results_df['forward_pct'], width=0.4, label='Forward Pass')
ax2.set_xlabel('Code Distance')
ax2.set_ylabel('Time (%)')
ax2.set_title('Time Distribution by Step')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Throughput vs distance
ax3 = axes[1, 0]
ax3.plot(throughput_df['distance'], throughput_df['samples_per_sec'], 'o-', linewidth=2, markersize=8, color='green')
ax3.set_xlabel('Code Distance')
ax3.set_ylabel('Samples/sec')
ax3.set_title('Inference Throughput')
ax3.grid(True, alpha=0.3)

# Plot 4: Number of detectors vs distance
ax4 = axes[1, 1]
ax4.bar(throughput_df['distance'], throughput_df['num_detectors'], color='coral')
ax4.set_xlabel('Code Distance')
ax4.set_ylabel('Number of Detectors')
ax4.set_title('Detector Count Scaling (d² growth)')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

In [ ]:
print("="*60)
print("INFERENCE SPEED SUMMARY")
print("="*60)

print(f"\nModel: {MODEL_NAME}")
print(f"Device: {device}")
print(f"Samples tested: {NUM_SAMPLES}")

print(f"\n--- Timing Summary ---")
print(f"{'Distance':<10} {'Total (ms)':<12} {'Coord %':<10} {'Throughput':<12}")
print("-" * 45)
for i, row in results_df.iterrows():
    tp = throughput_df[throughput_df['distance'] == row['distance']]['samples_per_sec'].values[0]
    print(f"{row['distance']:<10} {row['total_ms']:<12.3f} {row['coord_pct']:<10.1f} {tp:<12.0f}")

print(f"\n--- Key Findings ---")
# Calculate scaling factor
d3_time = results_df[results_df['distance'] == 3]['total_ms'].values[0]
d13_time = results_df[results_df['distance'] == 13]['total_ms'].values[0]
scaling = d13_time / d3_time

avg_coord_pct = results_df['coord_pct'].mean()

print(f"- Time scaling d=3 to d=13: {scaling:.1f}x")
print(f"- Average coord conversion time: {avg_coord_pct:.1f}% of total")
print(f"- Bottleneck: {'Coordinate Conversion' if avg_coord_pct > 50 else 'Forward Pass'}")

## Potential Optimizations

Based on the analysis, here are potential optimization targets:

1. **If Coordinate Conversion is slow:**
   - Pre-compute coordinate mappings for all distances
   - Use torch.compile() on the conversion function
   - Batch multiple distances together

2. **If Forward Pass is slow:**
   - Use torch.compile() on the model
   - Reduce model size (fewer hidden units)
   - Use mixed precision (FP16/BF16)

3. **General optimizations:**
   - Increase batch size if memory allows
   - Use CUDA graphs for repeated inference
   - Profile with torch.profiler for detailed breakdown